In [1]:
# ── Cell 1 · Imports & version check ──────────────────────────────────────
import pandas as pd
import numpy as np
import scipy
import matplotlib
import seaborn as sns
import yfinance as yf
import statsmodels

print("pandas      :", pd.__version__)
print("numpy       :", np.__version__)
print("scipy       :", scipy.__version__)
print("matplotlib  :", matplotlib.__version__)
print("seaborn     :", sns.__version__)
print("yfinance    :", yf.__version__)
print("statsmodels :", statsmodels.__version__)
print("\nAll imports OK ✓")

pandas      : 3.0.3
numpy       : 2.4.6
scipy       : 1.17.1
matplotlib  : 3.10.9
seaborn     : 0.13.2
yfinance    : 1.4.0
statsmodels : 0.14.6

All imports OK ✓


In [2]:
# ── Cell 2 · Define tickers & download raw price data ─────────────────────

# Benchmark and diversification candidates
TICKERS = [
    "SPY",   # S&P 500 — main benchmark
    "QQQ",   # Nasdaq 100 — tech concentrated
    "IWM",   # Russell 2000 — small caps
    "GLD",   # Gold — uncorrelated asset
    "TLT",   # Long-term US bonds — classic negative correlation
    "VNQ",   # Real estate
    "EEM",   # Emerging markets
    # Known portfolio holdings
    "FXAIX", # Fidelity 500 Index Fund
    "AAPL",  # Apple
    "MSFT",  # Microsoft
    "VFFVX", # Vanguard Target Retirement 2055
]

START_DATE = "2015-01-01"
END_DATE   = "2025-12-31"

print(f"Downloading data for {len(TICKERS)} tickers...")
print(f"Period: {START_DATE} → {END_DATE}\n")

raw_prices = yf.download(
    tickers=TICKERS,
    start=START_DATE,
    end=END_DATE,
    auto_adjust=True,
)["Close"]

print(f"\nShape: {raw_prices.shape}")
print(f"Columns: {list(raw_prices.columns)}")
print(f"\nFirst rows:")
raw_prices.head()

Period: 2015-01-01 → 2025-12-31



[*********************100%***********************]  11 of 11 completed



Shape: (2765, 11)
Columns: ['AAPL', 'EEM', 'FXAIX', 'GLD', 'IWM', 'MSFT', 'QQQ', 'SPY', 'TLT', 'VFFVX', 'VNQ']

First rows:


Ticker,AAPL,EEM,FXAIX,GLD,IWM,MSFT,QQQ,SPY,TLT,VFFVX,VNQ
2015-01-02,24.192602,30.355053,58.758236,114.080002,102.786980,39.681747,94.665085,170.125031,93.367317,23.635958,52.549862
2015-01-05,23.511061,29.814816,57.685390,115.800003,101.412796,39.316826,93.276443,167.052597,94.833961,23.251276,52.837479
2015-01-06,23.513277,29.689533,57.177155,117.120003,99.658371,38.739780,92.025795,165.479126,96.542671,23.058931,53.361580
2015-01-07,23.842979,30.331562,57.854759,116.430000,100.885582,39.231972,93.212128,167.541153,96.351982,23.280863,54.179665
2015-01-08,24.759083,30.848314,58.895416,115.940002,102.596840,40.386108,94.996140,170.514221,95.075943,23.628563,54.384201


In [3]:
# ── Cell 3 · Data quality check ───────────────────────────────────────────

print("=== Shape ===")
print(f"{raw_prices.shape[0]} trading days × {raw_prices.shape[1]} tickers\n")

print("=== Date range ===")
print(f"Start : {raw_prices.index[0].date()}")
print(f"End   : {raw_prices.index[-1].date()}\n")

print("=== Missing values ===")
missing = raw_prices.isnull().sum()
print(missing)
print(f"\nTotal missing: {missing.sum()}")

print("\n=== Basic stats ===")
raw_prices.describe().round(2)

=== Shape ===
2765 trading days × 11 tickers

=== Date range ===
Start : 2015-01-02
End   : 2025-12-30

=== Missing values ===
Ticker
AAPL     0
EEM      0
FXAIX    0
GLD      0
IWM      0
MSFT     0
QQQ      0
SPY      0
TLT      0
VFFVX    0
VNQ      0
dtype: int64

Total missing: 0

=== Basic stats ===


Ticker,AAPL,EEM,FXAIX,GLD,IWM,MSFT,QQQ,SPY,TLT,VFFVX,VNQ
count,2765.00,2765.00,2765.00,2765.00,2765.00,2765.00,2765.00,2765.00,2765.00,2765.00,2765.00
mean,106.07,37.47,117.02,165.67,159.20,205.16,264.42,336.89,100.78,37.75,69.72
std,74.13,6.50,49.00,59.92,41.60,142.59,144.03,140.22,15.83,11.28,12.89
min,20.57,22.63,53.46,100.50,83.12,34.28,89.47,154.56,74.27,20.88,45.22
25%,36.58,33.43,75.77,120.90,124.89,68.48,137.98,218.90,89.67,28.59,58.53
50%,88.30,36.92,104.05,159.33,152.25,191.12,240.74,299.88,96.13,35.54,68.81
75%,168.32,41.18,146.61,180.60,196.20,310.98,359.20,421.92,108.58,45.52,80.26
max,285.66,55.22,240.00,416.74,256.48,538.66,634.15,688.50,143.79,66.72,98.08


In [4]:
# ── Cell 4 · Save raw data to disk ────────────────────────────────────────

import os

# data/raw is in .gitignore — safe to store here
output_path = "../data/raw/prices_raw.csv"

raw_prices.to_csv(output_path)

print(f"Saved to: {output_path}")
print(f"File size: {os.path.getsize(output_path) / 1024:.1f} KB")

Saved to: ../data/raw/prices_raw.csv
File size: 571.1 KB


In [5]:
# ── Cell 5 · Calculate log returns & save ─────────────────────────────────

# Log returns: ln(P_t / P_t-1)
# Why log returns?
# 1. Additive over time (daily returns sum to total return)
# 2. Better statistical properties (closer to normal distribution)
# 3. Standard in quantitative finance

log_returns = np.log(raw_prices / raw_prices.shift(1)).dropna()

print(f"Log returns shape: {log_returns.shape}")
print(f"Date range: {log_returns.index[0].date()} → {log_returns.index[-1].date()}")
print(f"\nDaily log returns (first 3 rows):")
print(log_returns.head(3).round(4))

# Save
log_returns.to_csv("../data/raw/log_returns.csv")
print("\nSaved log_returns.csv ✓")

log_returns.describe().round(4)

Log returns shape: (2764, 11)
Date range: 2015-01-05 → 2025-12-30

Daily log returns (first 3 rows):
Ticker        AAPL     EEM   FXAIX     GLD     IWM    MSFT     QQQ     SPY  \
2015-01-05 -0.0286 -0.0180 -0.0184  0.0150 -0.0135 -0.0092 -0.0148 -0.0182   
2015-01-06  0.0001 -0.0042 -0.0088  0.0113 -0.0175 -0.0148 -0.0135 -0.0095   
2015-01-07  0.0139  0.0214  0.0118 -0.0059  0.0122  0.0126  0.0128  0.0124   

Ticker         TLT   VFFVX     VNQ  
2015-01-05  0.0156 -0.0164  0.0055  
2015-01-06  0.0179 -0.0083  0.0099  
2015-01-07 -0.0020  0.0096  0.0152  

Saved log_returns.csv ✓


Ticker,AAPL,EEM,FXAIX,GLD,IWM,MSFT,QQQ,SPY,TLT,VFFVX,VNQ
count,2764.0000,2764.0000,2764.0000,2764.0000,2764.0000,2764.0000,2764.0000,2764.0000,2764.0000,2764.0000,2764.0000
mean,0.0009,0.0002,0.0005,0.0005,0.0003,0.0009,0.0007,0.0005,-0.0000,0.0004,0.0002
std,0.0182,0.0129,0.0113,0.0093,0.0143,0.0169,0.0139,0.0112,0.0095,0.0095,0.0130
min,-0.1377,-0.1333,-0.1271,-0.0664,-0.1423,-0.1595,-0.1276,-0.1159,-0.0690,-0.1085,-0.1951
25%,-0.0073,-0.0065,-0.0037,-0.0046,-0.0070,-0.0067,-0.0050,-0.0037,-0.0057,-0.0035,-0.0057
50%,0.0010,0.0007,0.0007,0.0006,0.0008,0.0010,0.0012,0.0006,0.0004,0.0006,0.0007
75%,0.0100,0.0075,0.0058,0.0054,0.0082,0.0094,0.0078,0.0059,0.0056,0.0050,0.0069
max,0.1426,0.0775,0.0909,0.0479,0.0875,0.1329,0.1134,0.0999,0.0725,0.0761,0.0861
